In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

tickets = pd.read_csv("it_tickets.csv")
print(tickets.shape)
print(tickets.columns.tolist())
print(tickets.head())

(47837, 2)
['Document', 'Topic_group']
                                            Document    Topic_group
0  connection with icon icon dear please setup ic...       Hardware
1  work experience user work experience user hi w...         Access
2  requesting for meeting requesting meeting hi p...       Hardware
3  reset passwords for external accounts re expir...         Access
4  mail verification warning hi has got attached ...  Miscellaneous


In [2]:
print(tickets.columns[-1])  # or whichever column looks like the label
print(tickets.iloc[:, -1].value_counts())  # adjust index once you know the real column

Topic_group
Topic_group
Hardware                 13617
HR Support               10915
Access                    7125
Miscellaneous             7060
Storage                   2777
Purchase                  2464
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64


In [3]:
print(tickets.columns.tolist())
print(tickets.head(3))

['Document', 'Topic_group']
                                            Document Topic_group
0  connection with icon icon dear please setup ic...    Hardware
1  work experience user work experience user hi w...      Access
2  requesting for meeting requesting meeting hi p...    Hardware


In [4]:
# COunting occurence of unique values on topic_group column
print(tickets['Topic_group'].value_counts())

Topic_group
Hardware                 13617
HR Support               10915
Access                    7125
Miscellaneous             7060
Storage                   2777
Purchase                  2464
Internal Project          2119
Administrative rights     1760
Name: count, dtype: int64


In [5]:
# quality check functions:

# --- Schema validation ---
def check_schema(df, expected_columns):
    actual = set(df.columns)
    expected = set(expected_columns)
    missing = expected - actual
    extra = actual - expected
    return {
        "passed": len(missing) == 0,
        "missing_columns": list(missing),
        "extra_columns": list(extra),
    }

expected_cols = ['Document', 'Topic_group']

# --- Completeness / nulls ---
def check_nulls(df, threshold=0.02):
    null_rates = df.isnull().mean()
    flagged = null_rates[null_rates > threshold]
    return {
        "passed": len(flagged) == 0,
        "null_rates": null_rates.to_dict(),
        "flagged_columns": flagged.to_dict(),
    }

# --- Freshness (simulated, since Kaggle data has no real load timestamp) ---
def check_freshness(df, timestamp_col, max_age_hours=6):
    age_hours = (datetime.now() - pd.to_datetime(df[timestamp_col])).dt.total_seconds() / 3600
    stale = age_hours > max_age_hours
    return {
        "passed": stale.sum() == 0,
        "stale_row_count": int(stale.sum()),
        "stale_row_pct": float(stale.mean()),
    }

# --- Category distribution drift ---
def check_category_drift(baseline_df, new_df, category_col='Topic_group', threshold=0.05):
    baseline_dist = baseline_df[category_col].value_counts(normalize=True)
    new_dist = new_df[category_col].value_counts(normalize=True)
    combined = pd.DataFrame({'baseline': baseline_dist, 'new': new_dist}).fillna(0)
    combined['abs_diff'] = (combined['baseline'] - combined['new']).abs()
    drifted = combined[combined['abs_diff'] > threshold]
    return {
        "passed": len(drifted) == 0,
        "drifted_categories": drifted.to_dict('index'),
    }

In [6]:
from data_quality_utils import corrupt_dataset

clean_sample = tickets.sample(2000, random_state=1).reset_index(drop=True)
corrupted_sample = corrupt_dataset(clean_sample, seed=2)

clean_sample.to_csv("clean_sample.csv", index=False)

print("clean_sample shape:", clean_sample.shape)
print("corrupted_sample shape:", corrupted_sample.shape)
print("clean_sample columns:", clean_sample.columns.tolist())
print("corrupted_sample columns:", corrupted_sample.columns.tolist())

clean_sample shape: (2000, 2)
corrupted_sample shape: (2000, 2)
clean_sample columns: ['Document', 'Topic_group']
corrupted_sample columns: ['Document', 'Topic_group']


In [7]:
print("Nulls in clean_sample:\n", clean_sample.isnull().sum())
print("Nulls in corrupted_sample:\n", corrupted_sample.isnull().sum())

Nulls in clean_sample:
 Document       0
Topic_group    0
dtype: int64
Nulls in corrupted_sample:
 Document       92
Topic_group    92
dtype: int64


In [8]:
for test_seed in range(1, 20):
    result = corrupt_dataset(clean_sample, seed=test_seed)
    if result.shape[1] != clean_sample.shape[1] or list(result.columns) != list(clean_sample.columns):
        print(f"seed={test_seed} -> triggered structural change, columns: {result.columns.tolist()}")

[corruption] renamed column: Document -> Document_v2
seed=4 -> triggered structural change, columns: ['Document_v2', 'Topic_group']
[corruption] dropped column: Document
seed=9 -> triggered structural change, columns: ['Topic_group']
[corruption] renamed column: Topic_group -> Topic_group_v2
seed=10 -> triggered structural change, columns: ['Document', 'Topic_group_v2']
[corruption] dropped column: Topic_group
seed=11 -> triggered structural change, columns: ['Document']
[corruption] dropped column: Document
seed=12 -> triggered structural change, columns: ['Topic_group']
[corruption] dropped column: Topic_group
seed=13 -> triggered structural change, columns: ['Document']
[corruption] renamed column: Topic_group -> Topic_group_v2
seed=14 -> triggered structural change, columns: ['Document', 'Topic_group_v2']
[corruption] dropped column: Topic_group
seed=18 -> triggered structural change, columns: ['Document']


In [9]:
corrupted_sample = corrupt_dataset(clean_sample, seed=4)

print("Corrupted columns:", corrupted_sample.columns.tolist())
print("Nulls:\n", corrupted_sample.isnull().sum())

[corruption] renamed column: Document -> Document_v2
Corrupted columns: ['Document_v2', 'Topic_group']
Nulls:
 Document_v2    104
Topic_group    102
dtype: int64


In [10]:
expected_cols = ['Document', 'Topic_group']

print("=== Clean sample ===")
print(check_schema(clean_sample, expected_cols))
print(check_nulls(clean_sample))

print("=== Corrupted sample (seed=4) ===")
print(check_schema(corrupted_sample, expected_cols))
print(check_nulls(corrupted_sample))

=== Clean sample ===
{'passed': True, 'missing_columns': [], 'extra_columns': []}
{'passed': True, 'null_rates': {'Document': 0.0, 'Topic_group': 0.0}, 'flagged_columns': {}}
=== Corrupted sample (seed=4) ===
{'passed': False, 'missing_columns': ['Document'], 'extra_columns': ['Document_v2']}
{'passed': False, 'null_rates': {'Document_v2': 0.052, 'Topic_group': 0.051}, 'flagged_columns': {'Document_v2': 0.052, 'Topic_group': 0.051}}


In [11]:
# Freshness check needs load_timestamp added to both samples first (Step 7)
rng = np.random.default_rng(3)
now = datetime.now()
clean_sample['load_timestamp'] = [now - timedelta(hours=float(rng.uniform(0, 2))) for _ in range(len(clean_sample))]
corrupted_sample['load_timestamp'] = [now - timedelta(hours=float(rng.uniform(0, 48))) for _ in range(len(corrupted_sample))]

print("=== Freshness ===")
print("Clean:", check_freshness(clean_sample, 'load_timestamp'))
print("Corrupted:", check_freshness(corrupted_sample, 'load_timestamp'))

# Drift check — compare corrupted sample's category mix against the full dataset baseline
print("=== Drift ===")
print(check_category_drift(tickets, corrupted_sample, category_col='Topic_group'))

=== Freshness ===
Clean: {'passed': np.True_, 'stale_row_count': 0, 'stale_row_pct': 0.0}
Corrupted: {'passed': np.False_, 'stale_row_count': 1718, 'stale_row_pct': 0.859}
=== Drift ===
{'passed': True, 'drifted_categories': {}}


### NLP

In [12]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

tickets['clean_text'] = tickets['Document'].apply(clean_text)

print(tickets[['Document', 'clean_text']].head(3))

                                            Document  \
0  connection with icon icon dear please setup ic...   
1  work experience user work experience user hi w...   
2  requesting for meeting requesting meeting hi p...   

                                          clean_text  
0  connection with icon icon dear please setup ic...  
1  work experience user work experience user hi w...  
2  requesting for meeting requesting meeting hi p...  


In [13]:
# Check if clean_text actually changed anything
changed = (tickets['Document'] != tickets['clean_text']).sum()
print(f"Rows where cleaning changed the text: {changed} out of {len(tickets)}")

Rows where cleaning changed the text: 190 out of 47837


In [14]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    tickets['clean_text'], tickets['Topic_group'],
    test_size=0.2, random_state=42, stratify=tickets['Topic_group']
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

Train size: 38269
Test size: 9568

Train class distribution:
Topic_group
Hardware                 0.284643
HR Support               0.228174
Access                   0.148946
Miscellaneous            0.147587
Storage                  0.058063
Purchase                 0.051504
Internal Project         0.044292
Administrative rights    0.036792
Name: proportion, dtype: float64


In [15]:
#base line model 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))
overall_f1 = f1_score(y_test, y_pred, average='weighted')
print(f"Weighted F1: {overall_f1:.3f}")

                       precision    recall  f1-score   support

               Access       0.92      0.88      0.90      1425
Administrative rights       0.88      0.66      0.75       352
           HR Support       0.87      0.88      0.88      2183
             Hardware       0.81      0.90      0.85      2724
     Internal Project       0.92      0.81      0.86       424
        Miscellaneous       0.84      0.83      0.83      1412
             Purchase       0.99      0.87      0.92       493
              Storage       0.95      0.84      0.89       555

             accuracy                           0.86      9568
            macro avg       0.89      0.83      0.86      9568
         weighted avg       0.87      0.86      0.86      9568

Weighted F1: 0.863


In [16]:
# running with uncleaned text (alrady known that data is mostly clean)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    tickets['Document'], tickets['Topic_group'],
    test_size=0.2, random_state=42, stratify=tickets['Topic_group']
)

pipeline_raw = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000))
])

pipeline_raw.fit(X_train_raw, y_train_raw)
y_pred_raw = pipeline_raw.predict(X_test_raw)

f1_raw = f1_score(y_test_raw, y_pred_raw, average='weighted')
print(f"F1 without cleaning (raw Document): {f1_raw:.3f}")
print(f"F1 with cleaning (clean_text):      {overall_f1:.3f}")

F1 without cleaning (raw Document): 0.863
F1 with cleaning (clean_text):      0.863


In [17]:
# error analysis
results = pd.DataFrame({'text': X_test, 'true': y_test, 'predicted': y_pred})
errors = results[results['true'] != results['predicted']]

print(f"Error rate: {len(errors) / len(results):.1%}")
print("\nMost common misclassification pairs (true -> predicted):")
print(errors.groupby(['true', 'predicted']).size().sort_values(ascending=False).head(10))

Error rate: 13.7%

Most common misclassification pairs (true -> predicted):
true                   predicted    
HR Support             Hardware         158
Miscellaneous          Hardware         130
Hardware               HR Support       111
Administrative rights  Hardware          92
Access                 Hardware          90
Hardware               Miscellaneous     85
Miscellaneous          HR Support        75
HR Support             Miscellaneous     55
Hardware               Access            48
Storage                Hardware          48
dtype: int64


In [18]:
# fix for class imbalance
pipeline_v2 = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
pipeline_v2.fit(X_train, y_train)
y_pred_v2 = pipeline_v2.predict(X_test)

print(classification_report(y_test, y_pred_v2))
f1_v2 = f1_score(y_test, y_pred_v2, average='weighted')
print(f"v1 F1 (unweighted): {overall_f1:.3f}")
print(f"v2 F1 (class_weight='balanced'): {f1_v2:.3f}")

                       precision    recall  f1-score   support

               Access       0.90      0.89      0.89      1425
Administrative rights       0.66      0.88      0.75       352
           HR Support       0.89      0.85      0.87      2183
             Hardware       0.87      0.80      0.84      2724
     Internal Project       0.77      0.93      0.84       424
        Miscellaneous       0.80      0.86      0.83      1412
             Purchase       0.92      0.90      0.91       493
              Storage       0.85      0.93      0.89       555

             accuracy                           0.85      9568
            macro avg       0.83      0.88      0.85      9568
         weighted avg       0.86      0.85      0.85      9568

v1 F1 (unweighted): 0.863
v2 F1 (class_weight='balanced'): 0.855


In [19]:
# data lineage mock-up
import json
from datetime import datetime

lineage_log = []

def log_lineage(dataset_name, source, row_count, upstream=None):
    lineage_log.append({
        "dataset_name": dataset_name,
        "source": source,
        "load_timestamp": datetime.now().isoformat(),
        "row_count": row_count,
        "upstream_dataset": upstream,
    })

log_lineage("raw_tickets", "kaggle_it_ticket_dataset.csv", len(tickets), upstream=None)
log_lineage("cleaned_tickets", "preprocessing_pipeline", len(tickets), upstream="raw_tickets")
log_lineage("classified_tickets", "nlp_classification_pipeline", len(y_pred), upstream="cleaned_tickets")

lineage_df = pd.DataFrame(lineage_log)
lineage_df.to_csv("lineage_log.csv", index=False)
print(lineage_df)

         dataset_name                        source  \
0         raw_tickets  kaggle_it_ticket_dataset.csv   
1     cleaned_tickets        preprocessing_pipeline   
2  classified_tickets   nlp_classification_pipeline   

               load_timestamp  row_count upstream_dataset  
0  2026-08-22T12:41:06.041432      47837             None  
1  2026-08-22T12:41:06.041561      47837      raw_tickets  
2  2026-08-22T12:41:06.041596       9568  cleaned_tickets  


In [20]:
# Trust score:
def calculate_trust_score(schema_result, null_result, freshness_result):
    score = 100
    if not schema_result['passed']:
        score -= 30
    null_rate_avg = sum(null_result['null_rates'].values()) / len(null_result['null_rates'])
    score -= min(null_rate_avg * 100, 30)
    if not freshness_result['passed']:
        score -= 20
    return max(round(score, 1), 0)

clean_score = calculate_trust_score(
    check_schema(clean_sample, expected_cols),
    check_nulls(clean_sample),
    check_freshness(clean_sample, 'load_timestamp')
)
corrupted_score = calculate_trust_score(
    check_schema(corrupted_sample, expected_cols),
    check_nulls(corrupted_sample),
    check_freshness(corrupted_sample, 'load_timestamp')
)
print(f"Clean dataset trust score: {clean_score}")
print(f"Corrupted dataset trust score: {corrupted_score}")


Clean dataset trust score: 100.0
Corrupted dataset trust score: 46.6


In [21]:
# scorecard
scorecard = pd.DataFrame([
    {"dataset": "clean_sample", "trust_score": clean_score, "schema_passed": True, "checked_at": datetime.now()},
    {"dataset": "corrupted_sample", "trust_score": corrupted_score, "schema_passed": False, "checked_at": datetime.now()},
])
scorecard.to_csv("quality_scorecard.csv", index=False)
print(scorecard)

            dataset  trust_score  schema_passed                 checked_at
0      clean_sample        100.0           True 2026-08-22 12:41:06.079585
1  corrupted_sample         46.6          False 2026-08-22 12:41:06.079590


### Simulated pipeline

In [22]:
results_log = []

for seed in range(1, 121):
    sample = corrupt_dataset(clean_sample, seed=seed)

    # Simulate load timestamps — vary the "staleness" pattern per pipeline
    # so freshness failures aren't all-or-nothing across the whole set
    rng = np.random.default_rng(seed)
    max_hours = rng.choice([2, 6, 12, 48])  # some pipelines fresher than others
    sample = sample.copy()
    sample['load_timestamp'] = [
        datetime.now() - timedelta(hours=float(rng.uniform(0, max_hours)))
        for _ in range(len(sample))
    ]

    schema_result = check_schema(sample, expected_cols)
    # In the 120-pipeline loop only:
    null_result = check_nulls(sample, threshold=0.06)  # explicit override for this batch-scale scenario
    # null_result = check_nulls(sample)
    freshness_result = check_freshness(sample, 'load_timestamp')
    trust = calculate_trust_score(schema_result, null_result, freshness_result)

    results_log.append({
        "pipeline_id": f"pipeline_{seed:03d}",
        "columns": ", ".join(sample.columns.tolist()),
        "schema_passed": schema_result['passed'],
        "null_check_passed": null_result['passed'],
        "avg_null_rate": round(sum(null_result['null_rates'].values()) / len(null_result['null_rates']), 4),
        "freshness_passed": freshness_result['passed'],
        "trust_score": trust,
    })

pipeline_results = pd.DataFrame(results_log)
pipeline_results.to_csv("pipeline_validation_results.csv", index=False)

print(pipeline_results['trust_score'].describe())
print()
print("Pipelines failing at least one check:",
      (pipeline_results[['schema_passed', 'null_check_passed', 'freshness_passed']] == False).any(axis=1).sum(),
      "/ 120")
print()
print("Schema failures:", (~pipeline_results['schema_passed']).sum())
print("Null-check failures:", (~pipeline_results['null_check_passed']).sum())
print("Freshness failures:", (~pipeline_results['freshness_passed']).sum())

[corruption] renamed column: Document -> Document_v2
[corruption] dropped column: Document
[corruption] renamed column: load_timestamp -> load_timestamp_v2
[corruption] dropped column: load_timestamp
[corruption] dropped column: Document
[corruption] dropped column: load_timestamp
[corruption] renamed column: Topic_group -> Topic_group_v2
[corruption] dropped column: load_timestamp
[corruption] dropped column: load_timestamp
[corruption] renamed column: Topic_group -> Topic_group_v2
[corruption] dropped column: Document
[corruption] dropped column: Document
[corruption] dropped column: Topic_group
[corruption] dropped column: load_timestamp
[corruption] dropped column: Document
[corruption] dropped column: load_timestamp
[corruption] dropped column: Topic_group
[corruption] renamed column: Topic_group -> Topic_group_v2
[corruption] renamed column: Document -> Document_v2
[corruption] renamed column: Document -> Document_v2
[corruption] renamed column: Document -> Document_v2
[corruptio

## Tranformer model

In [23]:
pip install transformers datasets torch scikit-learn --break-system-packages

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU found")

True NVIDIA GeForce GTX 1650


In [25]:
import torch
print(torch.__version__)

2.13.0+cu132


In [26]:
pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.13.0+cu132
Uninstalling torch-2.13.0+cu132:
  Successfully uninstalled torch-2.13.0+cu132
Found existing installation: torchvision 0.28.0+cu132
Uninstalling torchvision-0.28.0+cu132:
  Successfully uninstalled torchvision-0.28.0+cu132
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [27]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

Looking in indexes: https://download.pytorch.org/whl/cu132
  Using cached torch-2.13.0%2Bcu132-cp313-cp313-win_amd64.whl.metadata (39 kB)
  Using cached torchvision-0.28.0%2Bcu132-cp313-cp313-win_amd64.whl.metadata (5.7 kB)
Using cached torch-2.13.0%2Bcu132-cp313-cp313-win_amd64.whl (1918.0 MB)
Using cached torchvision-0.28.0%2Bcu132-cp313-cp313-win_amd64.whl (9.2 MB)

   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "still no GPU")

2.13.0+cu132
True
NVIDIA GeForce GTX 1650


In [29]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
tickets['label'] = le.fit_transform(tickets['Topic_group'])

train_df, test_df = train_test_split(
    tickets[['Document', 'label']], test_size=0.2, random_state=42, stratify=tickets['label']
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch['Document'], truncation=True, padding='max_length', max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(le.classes_)
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"weighted_f1": f1_score(labels, preds, average='weighted')}

args = TrainingArguments(
    output_dir="./ticket_classifier",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())

Map:   0%|          | 0/38269 [00:00<?, ? examples/s]

Map:   0%|          | 0/9568 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Weighted F1
1,0.435388,0.393547,0.869408
2,0.267923,0.393472,0.876248
3,0.147248,0.442921,0.889843


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Weighted F1
0.147248,0.393472,3,0.876248


{'eval_loss': 0.3934723138809204, 'eval_weighted_f1': 0.8762477850090331}


## New script for weighted loss for class imbalance, plus early stopping

In [33]:
import torch
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from transformers import EarlyStoppingCallback

# Compute class weights from the training labels — same "balanced" logic
# as sklearn's class_weight='balanced' used earlier on the TF-IDF model
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to("cuda")
print("Class weights:", dict(zip(le.classes_, class_weights.round(2))))

# Custom Trainer that applies weighted loss instead of the default unweighted CrossEntropyLoss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Fresh model — don't reuse the already-trained one from the previous run
model_weighted = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(le.classes_)
)

args_weighted = TrainingArguments(
    output_dir="./ticket_classifier_weighted",
    num_train_epochs=5,                      # allow more room, early stopping will cut it off
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer_weighted = WeightedTrainer(
    model=model_weighted, args=args_weighted,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],  # stop once val loss stops improving
)

trainer_weighted.train()
print(trainer_weighted.evaluate())

Class weights: {'Access': np.float64(0.84), 'Administrative rights': np.float64(3.4), 'HR Support': np.float64(0.55), 'Hardware': np.float64(0.44), 'Internal Project': np.float64(2.82), 'Miscellaneous': np.float64(0.85), 'Purchase': np.float64(2.43), 'Storage': np.float64(2.15)}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Weighted F1
1,0.431059,0.394884,0.862602
2,0.296416,0.430795,0.867077


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Weighted F1
0.296416,0.394884,2,0.862602


{'eval_loss': 0.3948837220668793, 'eval_weighted_f1': 0.8626018123104541}


In [34]:
from sklearn.metrics import classification_report
preds_weighted = np.argmax(trainer_weighted.predict(test_ds).predictions, axis=1)
print(classification_report(test_df['label'], preds_weighted, target_names=le.classes_))

                       precision    recall  f1-score   support

               Access       0.87      0.93      0.90      1425
Administrative rights       0.77      0.82      0.79       352
           HR Support       0.88      0.89      0.88      2183
             Hardware       0.89      0.80      0.84      2724
     Internal Project       0.92      0.86      0.89       424
        Miscellaneous       0.81      0.85      0.83      1412
             Purchase       0.92      0.92      0.92       493
              Storage       0.79      0.94      0.86       555

             accuracy                           0.86      9568
            macro avg       0.86      0.87      0.86      9568
         weighted avg       0.86      0.86      0.86      9568



In [35]:
from sklearn.metrics import classification_report

preds_unweighted = np.argmax(trainer.predict(test_ds).predictions, axis=1)
print(classification_report(test_df['label'], preds_unweighted, target_names=le.classes_))

                       precision    recall  f1-score   support

               Access       0.85      0.95      0.90      1425
Administrative rights       0.82      0.78      0.80       352
           HR Support       0.89      0.90      0.89      2183
             Hardware       0.88      0.84      0.86      2724
     Internal Project       0.88      0.89      0.88       424
        Miscellaneous       0.85      0.84      0.84      1412
             Purchase       0.94      0.90      0.92       493
              Storage       0.93      0.88      0.90       555

             accuracy                           0.88      9568
            macro avg       0.88      0.87      0.88      9568
         weighted avg       0.88      0.88      0.88      9568



In [7]:
import numpy, pandas
print("Local numpy version:", numpy.__version__)
print("Local pandas version:", pandas.__version__)

Local numpy version: 2.2.6
Local pandas version: 2.3.1
